# 02 · LightGBM (gradient boosting, the M5 winner-style model)

**Owner:** Elena · **Reads:** notebook 01 outputs · **Feeds:** notebook 06 (hybrid model, ensemble) and 05 (scoring)

One LightGBM model learns from **all** California products at once (a "global" model), the way the top M5 solutions did.

**Two runs**
| Run | Model name in files | What the model sees |
|---|---|---|
| Sales only | `lgbm` | Past sales (lags and rolling averages), weekday/month, product/department/category/store |
| With covariates | `lgbm_cov` | Everything above + price, price vs. the previous 28 days, price change, SNAP days, events, Christmas (today's price and calendar are known in advance; comparisons use past prices only) |

**How it avoids peeking:** every sales feature is at least **28 days old**. For a window starting on day *S*, even the last day of the window only uses sales up to day *S − 1*. So one model forecasts all 28 days directly (no feeding forecasts back in), and it is trained only on days before *S*. A new model is trained for every window.

**Seven windows per run:** 4 backtests (`bt_4` … `bt_1`), `val_1`, `val_2`, `test`, identical to 02a/03/04a/04b. The backtests are what the hybrid model and ensemble learn from in 06.

**Output:** `data/forecasts/<model>_<window>.parquet` with columns `unique_id, ds, model, window, forecast`, so 05 scores them automatically.

**Runtime:** CPU is fine (LightGBM doesn't use the GPU here). Start with `SAMPLE = True` (100 products, about a minute). Then `SAMPLE = False` for all of California. If Colab disconnects, just run everything again: finished windows are skipped.

## Notes for Prateek (running locally)

**Status (Nov 19, 4 PM):** Elena ran this on free Colab (about 20 min per window).
- `lgbm` (sales only): **all 7 windows done**, files already in Drive `data/forecasts/`.
- `lgbm_cov` (with prices/holidays): **not finished**. This is what's left.

**To run the rest on your machine:**
1. In Settings, set `DRIVE_DIR = r"C:\path\to\your\local\AML_Term_Project"` (the folder that contains `data/`, like your TimesFM run). Make sure your local `data/` has `ca_sales.parquet`, `ca_static.parquet` and `windows.json`.
2. Set `RUNS = ["lgbm_cov"]`, keep `SAMPLE = False`, `OVERWRITE = False`.
3. Run all. LightGBM uses every CPU core automatically. Optional: `pip install lightgbm` first.
4. Upload the 7 `lgbm_cov_<window>.parquet` files to Drive `data/forecasts/` (if any `lgbm_cov` files from Elena's Colab run are already there, yours can replace them: same code and settings).
5. Section 7 saves `figures/12_lgbm_feature_importance.png` locally; upload it to Drive `data/figures/` too.

**Leak fix already included:** price features use past prices only (`price_ratio_28` = price / average price over the previous 28 days), per the team's note. 01's full-history `price_ratio` is not used.


## 0 · Settings

In [ ]:
SAMPLE      = False      # True = 100-product smoke test; False = all ~12,200 California series
RUNS        = ["lgbm", "lgbm_cov"]   # sales only, with covariates
N_BACKTEST  = 4          # extra 28-day windows before val_1 (same as 02a/04a/04b)
OVERWRITE   = False      # True = redo windows that already have a forecast file (only for the runs listed in RUNS)
DRIVE_DIR   = None       # None = find the AML_Term_Project shortcut in My Drive automatically

TRAIN_DAYS  = 730        # train on the last 2 years before each window
LAGS        = [28, 35, 42, 49, 56, 364]          # days back (all >= 28, so no peeking)
ROLL        = [7, 14, 28, 56, 112]               # rolling averages of the 28-day-old sales
COVARIATES  = ["sell_price", "price_ratio_28", "price_change_7", "price_vs_max",
               "snap", "is_event", "event_type", "is_christmas"]

N_ROUNDS    = 600
PARAMS = {
    "objective": "tweedie",          # suits counts with many zeros (standard in M5)
    "tweedie_variance_power": 1.1,
    "learning_rate": 0.05,
    "num_leaves": 127,
    "min_data_in_leaf": 100,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,
    "lambda_l2": 0.1,
    "max_bin": 255,
    "seed": 42,
    "verbose": -1,
}

## 1 · Connect

In [ ]:
import os, glob, json, time
import numpy as np
import pandas as pd
import lightgbm as lgb

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Not running in Colab: set DRIVE_DIR in Settings to your local AML_Term_Project folder.")

if DRIVE_DIR is None:
    DRIVE_DIR = os.environ.get("AML_DIR")
if DRIVE_DIR is None:
    found = (glob.glob("/content/drive/MyDrive/AML_Term_Project")
             or glob.glob("/content/drive/MyDrive/*/AML_Term_Project"))
    assert found, "Can't find AML_Term_Project in My Drive. Add the shortcut, or set DRIVE_DIR."
    DRIVE_DIR = found[0]
DATA_DIR = f"{DRIVE_DIR}/data"
OUT_DIR  = f"{DATA_DIR}/forecasts_sample" if SAMPLE else f"{DATA_DIR}/forecasts"
FIG_DIR  = f"{DATA_DIR}/figures"
os.makedirs(OUT_DIR, exist_ok=True); os.makedirs(FIG_DIR, exist_ok=True)
print("Reading", DATA_DIR, "| writing", OUT_DIR, "| LightGBM", lgb.__version__)

## 2 · Load data and windows

In [ ]:
with open(f"{DATA_DIR}/windows.json") as f:
    wjson = json.load(f)
H = wjson["horizon"]

# Shared window list (identical to 02a/04a/04b): 4 backtests right before val_1, then val_1, val_2, test.
val1_start = pd.Timestamp(wjson["val_1"]["start"])
windows = {}
for k in range(N_BACKTEST, 0, -1):
    start = val1_start - pd.Timedelta(days=H * k)
    windows[f"bt_{k}"] = (start, start + pd.Timedelta(days=H - 1))
for name in ["val_1", "val_2", "test"]:
    windows[name] = (pd.Timestamp(wjson[name]["start"]), pd.Timestamp(wjson[name]["end"]))

# Only load the dates we need: training for the earliest window, plus the longest look-back
first_needed = min(s for s, _ in windows.values()) - pd.Timedelta(days=TRAIN_DAYS + max(LAGS) + max(ROLL) + 28)
cols = ["unique_id", "ds", "y", "sell_price", "snap", "is_event", "event_type", "is_christmas"]
df = pd.read_parquet(f"{DATA_DIR}/{'ca_sample100' if SAMPLE else 'ca_sales'}.parquet", columns=cols)
df["ds"] = pd.to_datetime(df["ds"])
df = df[df["ds"] >= first_needed]
df["unique_id"] = df["unique_id"].astype(str)
df = df.sort_values(["unique_id", "ds"]).reset_index(drop=True)

static = pd.read_parquet(f"{DATA_DIR}/ca_static.parquet")
static["unique_id"] = static["unique_id"].astype(str)
df = df.merge(static[["unique_id", "item_id", "dept_id", "cat_id", "store_id"]], on="unique_id", how="left")

# Rows must be one per product per day with no gaps, so "shift by 28 rows" = "28 days ago"
step = df.groupby("unique_id", sort=False)["ds"].diff().dropna()
assert (step == pd.Timedelta(days=1)).all(), "Gaps in the daily data: lags would be wrong"

print(f"{df['unique_id'].nunique():,} series, {len(df):,} rows, {df['ds'].min().date()} → {df['ds'].max().date()}")
pd.DataFrame({w: {"start": s.date(), "end": e.date()} for w, (s, e) in windows.items()}).T

## 3 · Features
Built once for all dates. Every sales feature looks at least 28 days back, so for any day in a window it only uses sales from before the window started.

| Feature | Meaning |
|---|---|
| `lag_28` … `lag_364` | Sales exactly that many days earlier (`lag_364` = same weekday last year) |
| `rmean_7` … `rmean_112` | Average daily sales over the 7…112 days ending 28 days ago |
| `rstd_28`, `rzero_28` | Spread of sales, and share of zero-sales days, over those 28 days |
| `dow`, `dom`, `month` | Weekday, day of month, month |
| `item_id`, `dept_id`, `cat_id`, `store_id` | Which product and store (categories) |
| covariates (`lgbm_cov` only) | Price, price vs. its average over the previous 28 days, % change vs. last week, price vs. its highest so far, SNAP, events, Christmas. All price features use past prices only (01's `price_ratio` uses the full-history average, so it isn't used here) |

In [ ]:
def rolling_by_product(x, ids, w):
    # Rolling mean, std and zero-share over the last w rows of each product, fully vectorised
    v = x.fillna(0).astype("float64")
    ok = x.notna().astype("float64")
    z = (x == 0).astype("float64")
    parts = {"s": v, "ss": v * v, "n": ok, "z": z}
    out = {}
    for k, s in parts.items():
        cs = s.groupby(ids, sort=False).cumsum()
        out[k] = cs - cs.groupby(ids, sort=False).shift(w).fillna(0)
    n = out["n"].replace(0, np.nan)
    mean = out["s"] / n
    var = (out["ss"] / n - mean ** 2) * n / (n - 1)
    return mean, np.sqrt(var.clip(lower=0)), out["z"] / n

t0 = time.time()
ids = df["unique_id"]
g = df.groupby("unique_id", sort=False)["y"]
for L in LAGS:
    df[f"lag_{L}"] = g.shift(L).astype("float32")

for w in ROLL:
    m, s, zr = rolling_by_product(df["lag_28"], ids, w)
    df[f"rmean_{w}"] = m.astype("float32")
    if w == 28:
        df["rstd_28"] = s.astype("float32")
        df["rzero_28"] = zr.astype("float32")

df["dow"] = df["ds"].dt.dayofweek.astype("int8")
df["dom"] = df["ds"].dt.day.astype("int8")
df["month"] = df["ds"].dt.month.astype("int8")

# Price features (prices are set in advance, so the window's own prices are allowed, same as chronos2_cov)
gp = df.groupby("unique_id", sort=False)["sell_price"]
df["price_change_7"] = (df["sell_price"] / gp.shift(7) - 1).astype("float32")
df["price_vs_max"] = (df["sell_price"] / gp.cummax()).astype("float32")
# Price vs. its average over the PREVIOUS 28 days. (01's price_ratio divides by the full-history
# average, which includes future prices, so we don't use it here.)
past_price_mean, _, _ = rolling_by_product(gp.shift(1), ids, 28)
df["price_ratio_28"] = (df["sell_price"] / past_price_mean).astype("float32")

for c in ["item_id", "dept_id", "cat_id", "store_id", "event_type"]:
    df[c] = df[c].astype(str).astype("category")

BASE_FEATS = ([f"lag_{L}" for L in LAGS] + [f"rmean_{w}" for w in ROLL] + ["rstd_28", "rzero_28"]
              + ["dow", "dom", "month", "item_id", "dept_id", "cat_id", "store_id"])
FEATURES = {"lgbm": BASE_FEATS, "lgbm_cov": BASE_FEATS + COVARIATES}
CATS = ["item_id", "dept_id", "cat_id", "store_id", "event_type"]
print(f"Features built in {time.time() - t0:.0f}s: {len(BASE_FEATS)} sales-only, {len(FEATURES['lgbm_cov'])} with covariates")

## 4 · Train and forecast one window
For a window starting on day *S*:
- **Training rows** = the `TRAIN_DAYS` days before *S* (actual sales of the window are never used)
- **Forecast rows** = the 28 window days, for every product with at least one day of history before *S* (same rule as the other notebooks)
- Forecasts below zero are set to zero

In [ ]:
importances = {}

def forecast_window(run, wname, start, end):
    feats = FEATURES[run]
    cats = [c for c in CATS if c in feats]
    train = df[(df["ds"] < start) & (df["ds"] >= start - pd.Timedelta(days=TRAIN_DAYS)) & df["lag_28"].notna()]
    ids = df.loc[df["ds"] < start, "unique_id"].unique()
    pred = df[(df["ds"] >= start) & (df["ds"] <= end) & df["unique_id"].isin(ids)]

    dtrain = lgb.Dataset(train[feats], label=train["y"], categorical_feature=cats, free_raw_data=True)
    model = lgb.train(PARAMS, dtrain, num_boost_round=N_ROUNDS)
    importances[(run, wname)] = pd.Series(model.feature_importance("gain"), index=feats)

    res = pred[["unique_id", "ds"]].copy()
    res["forecast"] = np.clip(model.predict(pred[feats]), 0, None).astype("float32")
    res["model"] = run
    res["window"] = wname
    return res[["unique_id", "ds", "model", "window", "forecast"]].reset_index(drop=True), len(train)

## 5 · Run everything (skips windows already saved)

In [ ]:
log_rows = []
for run in RUNS:
    for wname, (start, end) in windows.items():
        path = f"{OUT_DIR}/{run}_{wname}.parquet"
        if os.path.exists(path) and not OVERWRITE:
            print(f"skip  {run:8s} {wname:6s} (already saved)")
            continue
        t0 = time.time()
        res, n_train = forecast_window(run, wname, start, end)
        assert len(res) == res["unique_id"].nunique() * H, "every product should have exactly 28 days"
        res.to_parquet(path, index=False)
        secs = time.time() - t0
        n = res["unique_id"].nunique()
        print(f"done  {run:8s} {wname:6s} {n:6,} series  {n_train:11,} training rows  {secs:6.1f}s")
        log_rows.append({"model": run, "window": wname, "series": n, "train_rows": n_train,
                         "seconds": round(secs, 1), "device": "cpu", "sample": SAMPLE,
                         "run_at": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M")})

if log_rows:
    log_path = f"{OUT_DIR}/run_log.csv"
    new = pd.DataFrame(log_rows)
    new.to_csv(log_path, mode="a", header=not os.path.exists(log_path), index=False)
    display(new)

## 6 · Quick check
Not the official evaluation (that's notebook 05). Mean absolute error per window for both LightGBM runs, next to the 4-week-average baseline and Chronos-2 if their files are in the same folder, all scored on exactly the same products and days.

In [ ]:
actual = df[["unique_id", "ds", "y"]]
compare = RUNS + ["smean4", "chronos2_zs"]
rows = []
for wname in windows:
    m = actual
    for name in compare:
        p = f"{OUT_DIR}/{name}_{wname}.parquet"
        if os.path.exists(p):
            f = pd.read_parquet(p, columns=["unique_id", "ds", "forecast"])
            f["unique_id"] = f["unique_id"].astype(str); f["ds"] = pd.to_datetime(f["ds"])
            m = m.merge(f.rename(columns={"forecast": name}), on=["unique_id", "ds"])
    row = {"window": wname, "series": m["unique_id"].nunique()}
    for name in compare:
        if name in m:
            row[f"MAE {name}"] = (m[name] - m["y"]).abs().mean()
    rows.append(row)

check = pd.DataFrame(rows).set_index("window")
for name in RUNS:
    if f"MAE {name}" in check and "MAE smean4" in check:
        check[f"{name} vs smean4"] = (1 - check[f"MAE {name}"] / check["MAE smean4"]).map("{:.0%}".format)
check.round(3)

## 7 · What the model relies on
Share of total gain per feature, averaged over the 7 windows (only windows trained in this session). Useful for the blog: it shows whether prices and holidays actually matter.

In [ ]:
import matplotlib.pyplot as plt
if importances:
    fig, axes = plt.subplots(1, len(RUNS), figsize=(6 * len(RUNS), 6))
    for ax, run in zip(np.atleast_1d(axes), RUNS):
        imp = [s / s.sum() for (r, _), s in importances.items() if r == run]
        if not imp:
            ax.set_axis_off(); continue
        top = pd.concat(imp, axis=1).mean(axis=1).sort_values().tail(15)
        ax.barh(top.index, top.values, color=["#BF5700" if f in COVARIATES else "#1f4e79" for f in top.index])
        ax.set_title(f"{run}: top features (orange = covariate)", fontsize=10)
        ax.set_xlabel("share of total gain")
    plt.tight_layout()
    if not SAMPLE:
        plt.savefig(f"{FIG_DIR}/12_lgbm_feature_importance.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("Every window was already saved, so no models were trained this session (set OVERWRITE = True to retrain).")